In [0]:
display(
    dbutils.fs.ls(
        "file:/Workspace/Users/"
    )
)

path,name,size,modificationTime
file:/Workspace/Users/Groups/,Groups/,4096,1793498757080
file:/Workspace/Users/anuvishalp@gmail.com/,anuvishalp@gmail.com/,4096,1793498757080
file:/Workspace/Users/432ae511-a6ec-42a9-be55-f664c7e856b8/,432ae511-a6ec-42a9-be55-f664c7e856b8/,4096,1793498757080


In [0]:

# Step 1: Define the location of our CSV files

input_folder = (
    "file:/Workspace/Users/anuvishalp@gmail.com/"
    "Files/end-to-end data bricks/"
)

customers_path = input_folder + "ecommerce_customers.csv"
events_path = input_folder + "ecommerce_events.csv"

# Check whether Databricks can see the files
display(dbutils.fs.ls(input_folder))


path,name,size,modificationTime
file:/Workspace/Users/anuvishalp@gmail.com/Files/end-to-end data bricks/New File 2026-10-31 19:10:24.py,New File 2026-10-31 19:10:24.py,348,1793502437168
file:/Workspace/Users/anuvishalp@gmail.com/Files/end-to-end data bricks/ecommerce_events.csv,ecommerce_events.csv,1391,1793501898250
file:/Workspace/Users/anuvishalp@gmail.com/Files/end-to-end data bricks/ecommerce_customers.csv,ecommerce_customers.csv,984,1793501898261


In [0]:

# Step 2: Read customer and event CSV files

customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customers_path)
)

events_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(events_path)
)

# Display the data
display(customers_df)
display(events_df)


customer_id,email,signup_timestamp,country,membership_tier
1001,ALICE.JOHNSON@example.com,2024-01-15T09:30:00.000Z,USA,Gold
1002,bob.smith@example.com,2024-02-02T11:15:00.000Z,USA,Silver
1003,CHARLIE.BROWN@example.com,2024-02-18T14:05:00.000Z,Canada,Bronze
1004,diana.lee@example.com,2024-03-01T08:45:00.000Z,UK,Gold
1005,ethan.wilson@example.com,2024-03-12T16:20:00.000Z,India,Silver
1006,fatima.khan@example.com,2024-04-07T10:00:00.000Z,India,Gold
1007,george.miller@example.com,2024-04-21T12:30:00.000Z,Australia,Bronze
1008,hana.chen@example.com,2024-05-03T13:10:00.000Z,Canada,Silver
1009,isaac.davis@example.com,2024-05-19T17:40:00.000Z,USA,Gold
1010,julia.martin@example.com,2024-06-11T09:05:00.000Z,UK,Silver


event_id,customer_id,timestamp,event_type,transaction_amount
5001,1001,2024-09-01T10:00:00.000Z,purchase,129.99
5002,1002,2024-09-01T11:20:00.000Z,purchase,49.5
5003,1003,2024-09-02T09:15:00.000Z,view,0.0
5004,1004,2024-09-02T13:45:00.000Z,purchase,250.0
5005,1005,2024-09-03T15:30:00.000Z,purchase,79.99
5006,1006,2024-09-03T16:10:00.000Z,purchase,310.25
5007,1007,2024-09-04T08:50:00.000Z,cart,0.0
5008,1008,2024-09-04T12:05:00.000Z,purchase,65.75
5009,1009,2024-09-05T10:40:00.000Z,purchase,499.99
5010,1010,2024-09-05T14:25:00.000Z,view,0.0


In [0]:
# Step 3: Check row counts and column names

print("Customer records:", customers_df.count())
print("Event records:", events_df.count())

print("Customer columns:")
print(customers_df.columns)

print("Event columns:")
print(events_df.columns)

customers_df.printSchema()
events_df.printSchema()


Customer records: 15
Event records: 30
Customer columns:
['customer_id', 'email', 'signup_timestamp', 'country', 'membership_tier']
Event columns:
['event_id', 'customer_id', 'timestamp', 'event_type', 'transaction_amount']
root
 |-- customer_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_timestamp: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- membership_tier: string (nullable = true)

root
 |-- event_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- transaction_amount: double (nullable = true)



In [0]:

# Step 4: Create Bronze layer

from pyspark.sql.functions import current_timestamp

# Check which catalog and schema we are using
spark.sql("""
    SELECT current_catalog(), current_schema()
""").show(truncate=False)

# Add ingestion timestamp
bronze_customers_df = customers_df.withColumn(
    "ingested_at", current_timestamp()
)

bronze_events_df = events_df.withColumn(
    "ingested_at", current_timestamp()
)

# Save as Delta tables
bronze_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_customers")

bronze_events_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_events")

print("Bronze tables created successfully!")


+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|workspace        |default         |
+-----------------+----------------+

Bronze tables created successfully!


In [0]:

# Step 5: Verify Bronze tables

display(spark.table("bronze_customers"))

display(spark.table("bronze_events"))

print("Bronze customers:", spark.table("bronze_customers").count())
print("Bronze events:", spark.table("bronze_events").count())


customer_id,email,signup_timestamp,country,membership_tier,ingested_at
1001,ALICE.JOHNSON@example.com,2024-01-15T09:30:00.000Z,USA,Gold,2026-11-01T03:14:11.568Z
1002,bob.smith@example.com,2024-02-02T11:15:00.000Z,USA,Silver,2026-11-01T03:14:11.568Z
1003,CHARLIE.BROWN@example.com,2024-02-18T14:05:00.000Z,Canada,Bronze,2026-11-01T03:14:11.568Z
1004,diana.lee@example.com,2024-03-01T08:45:00.000Z,UK,Gold,2026-11-01T03:14:11.568Z
1005,ethan.wilson@example.com,2024-03-12T16:20:00.000Z,India,Silver,2026-11-01T03:14:11.568Z
1006,fatima.khan@example.com,2024-04-07T10:00:00.000Z,India,Gold,2026-11-01T03:14:11.568Z
1007,george.miller@example.com,2024-04-21T12:30:00.000Z,Australia,Bronze,2026-11-01T03:14:11.568Z
1008,hana.chen@example.com,2024-05-03T13:10:00.000Z,Canada,Silver,2026-11-01T03:14:11.568Z
1009,isaac.davis@example.com,2024-05-19T17:40:00.000Z,USA,Gold,2026-11-01T03:14:11.568Z
1010,julia.martin@example.com,2024-06-11T09:05:00.000Z,UK,Silver,2026-11-01T03:14:11.568Z


event_id,customer_id,timestamp,event_type,transaction_amount,ingested_at
5001,1001,2024-09-01T10:00:00.000Z,purchase,129.99,2026-11-01T03:14:17.817Z
5002,1002,2024-09-01T11:20:00.000Z,purchase,49.5,2026-11-01T03:14:17.817Z
5003,1003,2024-09-02T09:15:00.000Z,view,0.0,2026-11-01T03:14:17.817Z
5004,1004,2024-09-02T13:45:00.000Z,purchase,250.0,2026-11-01T03:14:17.817Z
5005,1005,2024-09-03T15:30:00.000Z,purchase,79.99,2026-11-01T03:14:17.817Z
5006,1006,2024-09-03T16:10:00.000Z,purchase,310.25,2026-11-01T03:14:17.817Z
5007,1007,2024-09-04T08:50:00.000Z,cart,0.0,2026-11-01T03:14:17.817Z
5008,1008,2024-09-04T12:05:00.000Z,purchase,65.75,2026-11-01T03:14:17.817Z
5009,1009,2024-09-05T10:40:00.000Z,purchase,499.99,2026-11-01T03:14:17.817Z
5010,1010,2024-09-05T14:25:00.000Z,view,0.0,2026-11-01T03:14:17.817Z


Bronze customers: 15
Bronze events: 30
